# FaithfulnessEvaluator usage

Faithfulness asks whether factual claims in generated `output` are supported by authoritative `context` (`output → context`). One judge call identifies and classifies claims; Python computes the supported-claim fraction. It requires `context + output`; optional `input` is descriptive and is not sent to the Faithfulness judge.

In [ ]:
from idp_eval import (
    EvaluationCase,
    EvaluationFramework,
    FaithfulnessEvaluator,
    create_azure_judge,
)
from idp_eval.judges import AzureJudgeConfig

## Configure a judge

Use placeholders only. Production applications normally populate this config from their settings/secrets layer. Evaluators are backend-independent; `create_gateway_judge(config=gateway_config)` can be used instead. See the setup guide and backend latency notebook for backend configuration.

In [ ]:
azure_config = AzureJudgeConfig(
    model="your-azure-deployment",
    azure_endpoint="https://your-resource.openai.azure.com",
    tenant_id="your-tenant-id",
    client_id="your-client-id",
    client_secret="your-client-secret",
    api_version="2024-12-01-preview",
    timeout=180,
    proxy_url=None,
    verify_ssl=True,
    reasoning_effort=None,
)
judge = create_azure_judge(config=azure_config)

Structured values are rendered generically: dictionary keys become readable labels, lists become bullets, and nesting is preserved. `verbose=True` exposes each supported or unsupported claim, its deterministic score, and its reason.

In [ ]:
case = EvaluationCase(
    input="Answer the customer using the supplied cancellation policy.",
    context={
        "cancellation_window_hours": 24,
        "refund_timeline": "5 business days",
    },
    output="""
    Customers may cancel within 24 hours.
    Refunds are issued instantly.
    """,
)
framework = EvaluationFramework(
    evaluators=[FaithfulnessEvaluator(judge, verbose=True)],
)
result = framework.evaluate(case)["faithfulness"]
{
    "score": result.score,
    "label": result.label,
    "explanation": result.explanation,
    "claims": result.details["claims"],
}

## Optional async equivalent

Jupyter supports top-level `await`; this is an alternative execution example and reuses the same setup.

In [ ]:
async_result = await framework.a_evaluate(case)
async_result["faithfulness"]

In [ ]:
judge.close()